# LSTM Auto-encoder (The sequence detection machine)

## It detects a sequence in the temporal pattern of what a normal array of days looks like. Then it tries to recreate those days in a similar fashion. If the pattern it created deviates from the original data, that is the "anomaly" I'm looking for. 

Some issue with this model and my data structure is that so far I have been using flat matrix of data, which is completely fine for statistics and isolation forest but does not work well here. It requres all the share data + the 5 features I created. Which isn't present in a single table so far. 

So first I must re-organize all the data points for this model to learn "What a normal might be?"

## Choice of data 

. To learn from: 2015-2019 data (Considered calmer economically)
. Testing data: 2015-2025 data (To compare with all the other models)

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

DATA_DIR = Path("../data")
features = pd.read_parquet(DATA_DIR / "features_clean.parquet")

feature_cols = ["ret_zscore", "vol_5", "vol_20", "vol_60", "vol_zscore"]
print("Loaded:", features.shape)
print("CUDA available:", torch.cuda.is_available())
features[feature_cols].head()

Loaded: (1184191, 5)
CUDA available: True


ret_zscore     vol_5    vol_20    vol_60  vol_zscore
Date       Ticker                                                      
2015-03-31 A        -0.265173  0.010081  0.012668  0.014717   -0.410469
           AAPL     -0.938536  0.020079  0.014414  0.017461   -0.650943
           ABBV      0.313154  0.015403  0.019325  0.020973   -0.017244
           ABT      -1.348182  0.009937  0.011308  0.011799    0.387669
           ACGL     -2.030660  0.011328  0.009421  0.008541    2.016447

In [2]:

def make_sequences(stock_features, window=30):
    """
    stock_features: 2D array (days, 5 features) for ONE stock, in date order
    Returns: 3D array (num_sequences, window, 5)
             each sequence is `window` consecutive days of features
    """
    sequences = []
    for i in range(len(stock_features) - window + 1):
        seq = stock_features[i : i + window]   # days i to i+window-1
        sequences.append(seq)
    return np.array(sequences)

## First testing it on a small sample of Apple 

In [3]:
feature_cols = ["ret_zscore", "vol_5", "vol_20", "vol_60", "vol_zscore"]

aapl = features.xs("AAPL", level="Ticker").sort_index()
aapl_arr = aapl[feature_cols].values
print("AAPL feature array shape:", aapl_arr.shape)

aapl_seqs = make_sequences(aapl_arr, window=30)
print("AAPL sequences shape:", aapl_seqs.shape)

AAPL feature array shape: (2456, 5)
AAPL sequences shape: (2427, 30, 5)


## Since it was able to work on 1 stock. I'll run all 499 stocks. 

I cannot just import all the stocks and run this 2D to 3D data conversion as that would combine different stocks together tainting the data. I have to stack each ticker on top of one another and then under each ticker the data can be stored by date and their respective features. 

In [4]:
all_sequences = []
sequence_index = []   # track which (date, ticker) each sequence ENDS on

for ticker in features.index.get_level_values("Ticker").unique():
    stock = features.xs(ticker, level="Ticker").sort_index()
    arr = stock[feature_cols].values
    if len(arr) < 30:          # too short to form even one sequence
        continue
    seqs = make_sequences(arr, window=30)
    all_sequences.append(seqs)
    # the endpoint date of each sequence (day 30 onward)
    sequence_index.extend(
        [(d, ticker) for d in stock.index[29:]]
    )

X_seq = np.concatenate(all_sequences, axis=0)
print("Full sequence array shape:", X_seq.shape)
print("Index entries:", len(sequence_index))

Full sequence array shape: (1169720, 30, 5)
Index entries: 1169720


# Now splitting the data into learning and testing structures (PS testing structure is done)

In [6]:
seq_dates = pd.to_datetime([d for (d, t) in sequence_index])

train_mask = (seq_dates >= pd.Timestamp("2015-01-01")) & (seq_dates <= pd.Timestamp("2019-12-31"))

X_train = X_seq[train_mask]
print("Training sequences (2015-2019):", X_train.shape)
print("Total sequences (score everything):", X_seq.shape)

Training sequences (2015-2019): (550594, 30, 5)
Total sequences (score everything): (1169720, 30, 5)


## Identifying the NaNs because the model cannot learn from NaNs and will choke on them. 

In [7]:
print("NaNs in training data:", np.isnan(X_train).sum())
print("NaNs in full data:", np.isnan(X_seq).sum())

NaNs in training data: 0
NaNs in full data: 0


## Perfect, everything is ready to run now. 

# The LSTM Autoencoder:

## First the autoencoder: It basically learns to compress and decompress the data. 

In [8]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features=5, hidden_size=32, seq_len=30):
        super().__init__()
        self.seq_len = seq_len
        self.hidden_size = hidden_size

        # ENCODER: reads the 30-day sequence, compresses to one hidden vector
        self.encoder = nn.LSTM(n_features, hidden_size, batch_first=True)

        # DECODER: expands the hidden vector back into a 30-day sequence
        self.decoder = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        # x shape: (batch, 30, 5)
        _, (hidden, _) = self.encoder(x)          # encode → final hidden state
        # repeat that hidden vector 30 times, one per timestep to reconstruct
        latent = hidden[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
        decoded, _ = self.decoder(latent)          # decode → 30-day sequence
        reconstruction = self.output_layer(decoded)  # map back to 5 features
        return reconstruction

model = LSTMAutoencoder().to(device)
print(model)
print("Device:", device)

LSTMAutoencoder(
  (encoder): LSTM(5, 32, batch_first=True)
  (decoder): LSTM(32, 32, batch_first=True)
  (output_layer): Linear(in_features=32, out_features=5, bias=True)
)
Device: cuda


## It's using GPU to learn which makes it faster. It encoded everything and decoded all 32 batches. 

> Good to proceed

## Next we set up the machine. Setting the parameters and how the data will be fed. 

> It will be fed in small batches so it does not overwhelm my GPU. 
> Next it measures loses for data during reconstruction and optimizer tinkers with the weights so each repetition of reconstruction creates lesser data losses
> lr = 1e-3 is the standard weight adjusting parameter

In [9]:
# Convert training data to a PyTorch tensor on the GPU
X_train_t = torch.tensor(X_train, dtype=torch.float32)

# DataLoader feeds data in batches rather than all 550k at once
from torch.utils.data import DataLoader, TensorDataset
train_loader = DataLoader(
    TensorDataset(X_train_t),
    batch_size=256,
    shuffle=True,
)

loss_fn = nn.MSELoss()                                  # how we measure reconstruction error
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # how we adjust weights

## Running 10 loops for data to compress and reconstruct each iteration showing the missing data value:

In [10]:
n_epochs = 10

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0.0
    for (batch,) in train_loader:
        batch = batch.to(device)

        reconstruction = model(batch)        # 1. forward: reconstruct the batch
        loss = loss_fn(reconstruction, batch)  # 2. measure how wrong it was

        optimizer.zero_grad()                # 3. clear old gradients
        loss.backward()                      # 4. compute how to adjust each weight
        optimizer.step()                     # 5. nudge weights toward less-wrong

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{n_epochs}  —  avg reconstruction loss: {avg_loss:.6f}")

Epoch 1/10  —  avg reconstruction loss: 0.315871
Epoch 2/10  —  avg reconstruction loss: 0.271624
Epoch 3/10  —  avg reconstruction loss: 0.257254
Epoch 4/10  —  avg reconstruction loss: 0.244453
Epoch 5/10  —  avg reconstruction loss: 0.233844
Epoch 6/10  —  avg reconstruction loss: 0.225488
Epoch 7/10  —  avg reconstruction loss: 0.219530
Epoch 8/10  —  avg reconstruction loss: 0.212496
Epoch 9/10  —  avg reconstruction loss: 0.206103
Epoch 10/10  —  avg reconstruction loss: 0.199663


## As we can see, the data loss is clearly converging towards a lower point. This shows a healthy data reconstruction rate. Now I can keep running these untin Epoch = N {N > 10} but the returns in losses aren't massive. So it would just be trading off my time for a very small return. This is where I stop, for future comparision I might go for a much lesser losses to have a much defined trained model. 

For now, this shows a clear trend of reconstruction that I am content with. 

# Running it for all the data points: 

## 

In [11]:
model.eval()   # switch to evaluation mode (turns off training-only behaviors)

X_all_t = torch.tensor(X_seq, dtype=torch.float32)
all_loader = DataLoader(TensorDataset(X_all_t), batch_size=256, shuffle=False)

recon_errors = []
with torch.no_grad():                      # no gradient tracking → faster, less memory
    for (batch,) in all_loader:
        batch = batch.to(device)
        reconstruction = model(batch)
        # per-sequence error: average squared diff across all 30×5 values in each sequence
        err = ((reconstruction - batch) ** 2).mean(dim=(1, 2))
        recon_errors.extend(err.cpu().numpy())

recon_errors = np.array(recon_errors)
print("Reconstruction errors computed:", recon_errors.shape)
print("Should match sequence count:", X_seq.shape[0])

Reconstruction errors computed: (1169720,)
Should match sequence count: 1169720


In [12]:
print("Min error:   ", recon_errors.min())
print("Median error:", np.median(recon_errors))
print("Mean error:  ", recon_errors.mean())
print("Max error:   ", recon_errors.max())
print("99th pctile: ", np.percentile(recon_errors, 99))

Min error:    0.02374917
Median error: 0.1951775
Mean error:   0.19852388
Max error:    0.54840285
99th pctile:  0.3398534


In [13]:
torch.save(model.state_dict(), DATA_DIR / "lstm_autoencoder.pth")
np.save(DATA_DIR / "lstm_recon_errors.npy", recon_errors)
print("Saved model and errors.")

Saved model and errors.
